# ENADE 2021 — Mulheres em cursos de TI

Tratamento dos microdados do ENADE 2021 (INEP) para analisar a participação feminina
nos cursos de Tecnologia da Informação.

Os microdados trazem, em arquivos separados, a área de cada curso (`arq1`), o sexo (`arq5`)
e a idade (`arq6`) dos concluintes. Como cada curso pertence a uma única área, juntamos
essas informações pelo código do curso (`CO_CURSO`).

In [ ]:
import re
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW       = ROOT / 'data' / 'raw'
INEP      = RAW / 'inep'
PROCESSED = ROOT / 'data' / 'processed' / 'enade-2021'

## 1. Leitura dos microdados do INEP

- `arq1`: `CO_CURSO`, `CO_GRUPO` (área do curso)
- `arq5`: `CO_CURSO`, `TP_SEXO` (uma linha por estudante)
- `arq6`: `CO_CURSO`, `NU_IDADE` (uma linha por estudante)

In [ ]:
arq1 = pd.read_csv(INEP / 'microdados_2021_arq1.txt', sep=';', encoding='latin-1',
                   usecols=['CO_CURSO', 'CO_GRUPO'])
arq5 = pd.read_csv(INEP / 'microdados_2021_arq5.txt', sep=';', encoding='latin-1',
                   usecols=['CO_CURSO', 'TP_SEXO'])
arq6 = pd.read_csv(INEP / 'microdados_2021_arq6.txt', sep=';', encoding='latin-1',
                   usecols=['CO_CURSO', 'NU_IDADE'])
print(arq1.shape, arq5.shape, arq6.shape)

## 2. Associa a área a cada estudante via `CO_CURSO`

Montamos o mapa `CO_CURSO → CO_GRUPO` (cada curso tem uma única área) e aplicamos
no arquivo de sexo.

In [ ]:
course2group = arq1.drop_duplicates('CO_CURSO').set_index('CO_CURSO')['CO_GRUPO']
assert (arq1.groupby('CO_CURSO')['CO_GRUPO'].nunique() == 1).all()

arq5['CO_GRUPO'] = arq5['CO_CURSO'].map(course2group).astype('Int64').astype(str)
arq5 = arq5[arq5['TP_SEXO'].isin(['F', 'M'])]
arq5.head()

## 3. Nomes das áreas (dicionário oficial do INEP)

Áreas de TI/Computação: `72` Tec. ADS, `79` Tec. Redes, `4004` Ciência da Computação
(Bacharelado), `4005` Ciência da Computação (Licenciatura), `4006` Sistemas de Informação,
`6409` Tec. Gestão da TI.

In [ ]:
dic = pd.read_excel(INEP / 'dicionario_enade_2021.xlsx',
                    sheet_name='DICIONÁRIO DE VARIÁVEIS', header=None)

def tidy(s):
    minus = {'Da', 'De', 'Do', 'E', 'Em'}
    return ' '.join(w.lower() if w in minus else w for w in s.split())

AREA_NAME = {}
for i in range(26, len(dic)):
    if i > 26 and pd.notna(dic.iat[i, 0]):
        break
    for c in (4, 5):
        v = dic.iat[i, c]
        if pd.notna(v):
            m = re.match(r'\s*(\d+)\s*=\s*(.+)', str(v))
            if m:
                AREA_NAME[m.group(1)] = tidy(m.group(2).strip())

IT_CODES = ['72', '79', '4004', '4005', '4006', '6409']
{c: AREA_NAME[c] for c in IT_CODES}

## 4. Participação feminina por curso de TI

In [ ]:
ti = arq5[arq5['CO_GRUPO'].isin(IT_CODES)].copy()
ti['NOME_CURSO'] = ti['CO_GRUPO'].map(AREA_NAME)

part = ti.groupby('NOME_CURSO')['TP_SEXO'].value_counts().unstack(fill_value=0)
part['total'] = part['F'] + part['M']
part['%F'] = (part['F'] / part['total'] * 100).round(1)
part = part.sort_values('%F')
print('Women in IT:', int(part['F'].sum()),
      '| Men:', int(part['M'].sum()),
      f"| overall %F = {part['F'].sum()/part['total'].sum()*100:.1f}%")
part

## 5. Faixa etária dos concluintes de TI

Os microdados não vinculam sexo e idade por indivíduo, então a faixa etária considera
todos os concluintes de TI.

In [ ]:
arq6['CO_GRUPO'] = arq6['CO_CURSO'].map(course2group).astype('Int64').astype(str)
it_ages = arq6[arq6['CO_GRUPO'].isin(IT_CODES)].copy()
it_ages['AGE_GROUP'] = pd.cut(it_ages['NU_IDADE'],
                              bins=[0, 17, 24, 30, 40, 200],
                              labels=['under 17', '18-24', '25-30', '31-40', '40+'])
it_ages['AGE_GROUP'].value_counts().sort_index()

## 6. Exportação (mulheres em TI)

In [ ]:
women_it = ti[ti['TP_SEXO'] == 'F'].copy()
women_it.to_csv(PROCESSED / 'Enade2021_IT_Women.csv', index=False)
women_it.to_excel(PROCESSED / 'Enade2021_IT_Women.xlsx', index=False)
print(f'{len(women_it)} women in IT exported to data/processed/enade-2021/ (.csv and .xlsx)')